# PyTorch NN Module



In [57]:
import torch
import torch.nn as nn

# inherits `nn.Module`, not nn
class Model(nn.Module):
    
    def __init__(self, num_features):
        
        super().__init__()
        
        # linear layer
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        
        # activation function
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        
        output = self.linear(features)
        
        output = self.sigmoid(output)
        
        return output # y_pred



In [58]:
# create dataset

features = torch.rand(10, 5) # 10 samples, 5 features each

# create model

model = Model(num_features=features.shape[1])

# call model for forward pass

# model.forward(features) -> this works, but better is to:

model(features) # this is the standard way of calling it
                # this automatically triggers the `forward()` method of the model
                # apparently there is a __call__() method in the background that calls forward() when we call the model like this
                # magic methods!



tensor([[0.5521],
        [0.5356],
        [0.5778],
        [0.5742],
        [0.6328],
        [0.6738],
        [0.6817],
        [0.5501],
        [0.5472],
        [0.5605]], grad_fn=<SigmoidBackward0>)

In [59]:
print(model.linear.weight) # access the weights of the linear layer
print(model.linear.bias) # access the bias of the linear layer

Parameter containing:
tensor([[ 0.2662,  0.1567, -0.2716, -0.4464,  0.3586]], requires_grad=True)
Parameter containing:
tensor([0.3214], requires_grad=True)


In [60]:
from torchinfo import summary

summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

We can basically just use the `nn.Sequential` module to create a simple feedforward neural network instead of writing all the steps manually.

e.g.

```python

import torch
import torch.nn as nn

# inherits `nn.Module`, not nn
class Model(nn.Module):
    
    def __init__(self, num_features):
        
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(in_features=num_features, out_features=10),
            nn.Linear(in_features=10, out_features=1)
        )
        
    def forward(self, features):
        
        output = self.network(features)
        
        return output # y_pred
```


# Modifying the neural network from the previous lec


# Part 1: Fetching the data

In [61]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [62]:
# Importing the dataset

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [63]:

# REMEMBER: when rerunning this cell, rerun the cell that imports the dataset first to get the columns back
# otherwise, you will get an error that the columns are not found because they have already been dropped

df.shape

# drop unneeded columns

df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [64]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# Part 2: Preprocessing and setup



We use `StandardScaler` to scale the data. It converts each entry into $z = \frac{x - \mu}{\sigma}$ where $\mu$ is the mean and $\sigma$ is the standard deviation. This means that the data will have a mean of 0 and a standard deviation of 1. (TODO: gotta learn the theory behind this!)

In [65]:
scaler = StandardScaler()

# first fit(learn mean and std from training data) and transform it, then transform the test data (don't fit it to the test data)
X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

X_test

array([[ 1.5772238 ,  1.30055483,  1.49263439, ...,  0.4985941 ,
        -1.00920156, -1.26444498],
       [-0.29511884,  0.5700076 , -0.15940845, ...,  1.40496488,
         2.41212784,  1.25447429],
       [ 0.84703858,  1.06622836,  0.88031005, ...,  1.07995313,
         1.02062313,  0.03664294],
       ...,
       [-0.1899949 , -0.67743626, -0.21694228, ..., -0.36840905,
        -1.41122828, -0.42807019],
       [-0.0990769 , -0.75095045, -0.14543595, ..., -0.67221111,
         0.68095157, -0.38834647],
       [ 0.57712576,  0.07148952,  0.52277839, ...,  0.58099145,
        -0.4233912 , -1.10337345]], shape=(114, 30))

We then use `LabelEncoder` to convert the labels into integers. This is because the labels are currently in string format and we need them in integer format to train the model.

In [66]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)

y_test = encoder.transform(y_test)

y_test

array([1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1])

All of the data are `numpy` arrays now. We will convert them into `torch` tensors.

In [67]:
X_train_tensor = torch.from_numpy(X_train).type(torch.float64)
X_test_tensor = torch.from_numpy(X_test).type(torch.float64)
y_train_tensor = torch.from_numpy(y_train).type(torch.long)
y_test_tensor = torch.from_numpy(y_test).type(torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([455, 30])
torch.Size([455])


# Defining the model and training loop

In [68]:

class MySimpleNN(nn.Module):
    def __init__(self, num_features):
        
        super().__init__()
        
        # there are 30 weights to learn (one for each feature) and 1 bias to learn
        # self.weights : torch.Tensor = torch.rand(X_train.shape[1], 1, dtype=torch.float64, requires_grad=True)
        # self.bias : torch.Tensor = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        self.sigmoid = nn.Sigmoid()
        
        
        
        
    def forward(self, features):
    
        out = self.linear(features)
        
        out = self.sigmoid(out)
        
        return out
    
    
loss_function = nn.BCELoss()
    
    
        


## Some important params

In [69]:
# needed for gradient descent or other form of optimization
from numpy import float64


learning_rate = torch.tensor(0.1)
# how many runs
epochs = 250


In [70]:
# create an instance of the model

model = MySimpleNN(X_train_tensor.shape[1])



import torch.optim as optim

# AWLAYS call this after creating the model, otherwise it will not know which parameters to optimize
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


print(model.linear.weight)
print(model.linear.bias)


# this training is done in a loop

for epoch in range(epochs):
    
    # f/w pass
    y_pred = model(X_train_tensor.type(torch.float32))
    
    # print(y_pred[:5])
    

    # loss calc
    # y_pred is [455] but y_train_tensor is [455,1], so we 
    # had to reshape the tensor
    # we use `.view()` here for that.
    # here (-1, 1) means that we want to keep the number of columns 1 and infer the number of rows
    loss = loss_function(y_pred, y_train_tensor.type(torch.float32).view(-1, 1))
    
    # print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

    # b/w pass, update weights and bias
    optimizer.zero_grad() 
    
    loss.backward()
    
    optimizer.step()

Parameter containing:
tensor([[ 0.1504, -0.0324,  0.1346,  0.0058, -0.0835,  0.0313, -0.1749,  0.0515,
         -0.1808, -0.0333,  0.1664,  0.1730,  0.1049, -0.0173, -0.0447,  0.1440,
          0.1206,  0.1809, -0.0221,  0.1756,  0.1786,  0.1162,  0.0109, -0.0150,
          0.1320,  0.1175, -0.0052,  0.0204, -0.1115, -0.0606]],
       requires_grad=True)
Parameter containing:
tensor([0.1302], requires_grad=True)


In [71]:
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[ 0.5704,  0.3909,  0.5448,  0.4380,  0.1394,  0.1100,  0.1272,  0.4995,
         -0.0105, -0.2295,  0.6339,  0.0756,  0.4635,  0.3889, -0.0455, -0.1705,
         -0.1131,  0.1124, -0.1283, -0.1663,  0.7485,  0.6109,  0.5411,  0.5338,
          0.6200,  0.3303,  0.3267,  0.5142,  0.3629,  0.0939]],
       requires_grad=True)
Parameter containing:
tensor([-0.4809], requires_grad=True)


## Model Evaluation

Note that the original data for `y_train_tensor` are all in 0 or 1. So we decide a "threshold" value. e.g. 0.5 for `y_pred_test`. If the value is above 0.5, we will consider it as 1. If the value is below 0.5, we will consider it as 0. This is a common practice in binary classification problems.

In [72]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor.type(torch.float32))
    
    y_pred = (y_pred > 0.7).type(torch.float32)
    
    # calculate accuracy
    
    accuracy = (y_pred == y_test_tensor).float().mean()
    
    print(accuracy)

tensor(0.5140)


## `torch.optim` module

Provides a variety of optimization algorithms (e.g. SGD, Adam, etc.) that can be used to update the model parameters during training. It also provides a convenient way to compute the gradients and update the parameters in a single step.

Handles weight update and gradient calculations automatically for us. 

When we pass the `model.parameters()` (which is an iterator over all the weights - like `nn.Linear`, `nn.Conv2d`, and biases(if they exist) in the model) to the optimizer, it will know which parameters to update during training.

